# Fine-Tuning Llama-3.2-3B-Instruct into a Feedback → Gherkin Engine (QLoRA)

**Goal:** turn a compact instruct model into a deterministic formatting compiler that converts messy
user feedback into `PROBLEM STATEMENT` / `USER STORY` / strict-Gherkin `ACCEPTANCE CRITERIA` —
zero conversational preamble.

| | |
|---|---|
| Base model | `unsloth/Llama-3.2-3B-Instruct` (4-bit) |
| Method | QLoRA via Unsloth (LoRA r=16, α=16, dropout=0) |
| Data | 200 validated pairs from `data/train.json` |
| Hardware | 1× T4 (Colab) — runtime ≈ 10–15 min |
| LR / epochs | 2e-4, 3 epochs, `adamw_8bit` |

**Runtime → Change runtime type → T4 GPU** before running. Unsloth requires an NVIDIA GPU —
this notebook does not run on CPU/free-tier Mac kernels.

In [ ]:
# %%capture
# Unsloth pulls compatible transformers/trl/xformers versions on Colab.
import torch; assert torch.cuda.is_available(), "Switch to a T4 GPU runtime (Runtime > Change runtime type)"
!pip install unsloth


## 1. Configuration

In [ ]:
# Keep these in sync with eval/run_eval.py and backend/app.py.
BASE_MODEL     = "unsloth/Llama-3.2-3B-Instruct"
HF_USERNAME    = "mrchrislau"             # Hugging Face username (adapter namespace)
ADAPTER_REPO   = f"{HF_USERNAME}/feedback-to-gherkin-lora"

# Dataset: local when running from a repo clone, else from GitHub raw.
import os
if os.path.exists("../data/train.json"):
    TRAIN_PATH, EVAL_PATH = "../data/train.json", "../data/eval.json"
else:
    RAW = "https://raw.githubusercontent.com/chris-lau/feedback-to-gherkin-engine/main/data"   # fallback only: used when not running from a repo clone
    TRAIN_PATH, EVAL_PATH = f"{RAW}/train.json", f"{RAW}/eval.json"

SYSTEM_PROMPT = (
    "You are a requirements formatting engine. Convert the raw user feedback into exactly three sections in this order: 'PROBLEM STATEMENT:', 'USER STORY:', 'ACCEPTANCE CRITERIA:'. The user story must follow the pattern 'As a ..., I want ..., So that ...'. Acceptance criteria must use strict Gherkin (Scenario:, Given, When, Then, And). Output only the three sections. No greetings, no explanations, no markdown."
)
print("Data:", TRAIN_PATH)


## 2. Load the base model in 4-bit

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = 2048,
    dtype          = None,      # auto: bf16 on T4-supported kernels, else fp16
    load_in_4bit   = True,      # QLoRA 4-bit quantization
)


## 3. Attach LoRA adapters (QLoRA config)

Target modules cover the full attention block plus the MLP projections.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",   # fits T4 comfortably
    random_state   = 3407,
)


## 4. Prepare the training dataset

In [ ]:
import json, urllib.request

def _load(path):
    if path.startswith("http"):
        with urllib.request.urlopen(path) as r:
            return json.loads(r.read())
    with open(path) as f:
        return json.load(f)

from datasets import Dataset

rows = _load(TRAIN_PATH)
def to_text(row):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": row["input"]},
        {"role": "assistant", "content": row["output"]},
    ]
    return {{"text": tokenizer.apply_chat_template(messages, tokenize=False)}}

ds = Dataset.from_list(rows).map(to_text)
print(len(ds), "training pairs")
print(ds[0]["text"][:600])


## 5. Train (SFT, responses-only masking)

The loss is computed **only on the assistant section**, so the model learns to *produce* the schema,
not to echo the messy input.

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    dataset_text_field          = "text",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 1,
    num_train_epochs            = 3,
    learning_rate               = 2e-4,
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "linear",
    warmup_steps                = 10,
    logging_steps               = 10,
    seed                        = 3407,
    output_dir                  = "outputs",
    report_to                   = "none",
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=args)

# Mask prompt tokens (Llama-3.2 chat markers) so loss applies to the response only.
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
        response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
    )
except ImportError:
    print("train_on_responses_only unavailable in this unsloth version; training on full sequence.")

import time
t0 = time.time()
trainer.train()
print(f"trained in {(time.time() - t0)/60:.1f} min")


## 6. Before / after sanity check

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

sample = _load(EVAL_PATH)[0]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": sample["input"]},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

def gen():
    return tokenizer.decode(
        model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.0, do_sample=False,
                       pad_token_id=tokenizer.eos_token_id)[0][inputs.shape[-1]:],
        skip_special_tokens=True,
    )

print("=" * 30, "BASE (adapter off)", "=" * 30)
with model.disable_adapter():
    print(gen())
print("=" * 30, "FINE-TUNED", "=" * 30)
print(gen())


## 7. Publish the LoRA adapter

In [ ]:
model.save_pretrained("lora-adapter")
tokenizer.save_pretrained("lora-adapter")

from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))        # add HF_TOKEN via the Colab secrets panel
except Exception:
    login()                                      # fallback: interactive prompt

model.push_to_hub(ADAPTER_REPO, private=False)
tokenizer.push_to_hub(ADAPTER_REPO, private=False)
print(f"pushed -> https://huggingface.co/{ADAPTER_REPO}")


## Next steps

1. `eval/run_eval.py` — benchmark base vs fine-tuned on the 50 held-out samples, produce `eval/results.json`.
2. `backend/app.py` — deploy the dual-endpoint Gradio Space (`predict_base` / `predict_finetuned`).
3. `demo/index.html` — embed the split-screen comparison in the portfolio site.

The adapter is ~100 MB; the base model stays cached, so the Space only downloads the delta.